# CLIFF WALKING

![alt text](./static/safer-path-sutton.png "Title")

In [ ]:
from collections import defaultdict
import numpy as np
import itertools

import gym

from src.utils.plots import plot_training_results

## Q-Learning

In [ ]:
class QLearning():

    def __init__(self, env, alpha: float, gamma: float , epsilon: float, init_factor: float = 0.001):
        
        self.env = env
        na = env.action_space.n
        self.Qvalues = defaultdict(lambda: np.random.randn(na) * init_factor) 
        self.alpha = alpha 
        self.epsilon = epsilon 
        self.gamma = gamma 
            
    def act(self, s):
        if np.random.rand() > self.epsilon:
            return np.argmax(self.Qvalues[s])
        return self.env.action_space.sample()
            
    def learn(self, s, a, r, s1, done=False): 
        qsa = self.Qvalues[s][a]
        if done:
            target = r 
        else:
            target = r + self.gamma * np.max(self.Qvalues[s1]) 
           
        self.Qvalues[s][a] = (1 - self.alpha) * qsa + self.alpha * target

        return (target - qsa) ** 2

#### Q-LEARNING PSEUDOCODE

##### Un poquito más de verbose

In [ ]:
class QLearning():

    def __init__(self, env, alpha: float, gamma: float , epsilon: float, init_factor: float = 0.001):
        
        self.env = env
        na = env.action_space.n
        self.Qvalues = defaultdict(lambda: np.random.randn(na) * init_factor) 
        self.alpha = alpha 
        self.epsilon = epsilon 
        self.gamma = gamma 
            
    def act(self, s):
        if np.random.rand() > self.epsilon:
            a = self.act_no_explore(s)
        else: 
            a = self.env.action_space.sample()
        return a

    def act_no_explore(self, s):
        return np.argmax(self.getQs(s))

    def getQs(self, s):
        return self.Qvalues[s]
            
    def learn(self, s, a, r, s1, done=False): 
        qsa = self.getQs(s)[a]
        if done:
            target = r 
        else:
            target = r + self.gamma * np.max(self.getQs(s1)) 
           
        self.Qvalues[s][a] = (1 - self.alpha) * qsa + self.alpha * target

        return (target - qsa) ** 2

In [ ]:
env = gym.make('CliffWalking-v0')
env.reset()

nepisodes = 300
print_every = 50

gamma = 0.99
alpha = 0.1
epsilon = 1
min_epsilon = 0.05
epsilon_decay = (1 - min_epsilon) / (nepisodes * 0.5)

agent = QLearning(env, alpha=alpha, gamma=gamma, epsilon=epsilon)

history_rewards = []
history_losses = []
for episode in range(nepisodes):
    episode_losses = []
    step = 0
    total_reward = 0
    s = env.reset()
    while True:
        a = agent.act(s)
        s1, r, done, _ = env.step(a)
        total_reward += gamma**step * r
        loss = agent.learn(s, a, r, s1, done)
        episode_losses.append(loss)
        s = s1
        step += 1
        if done:
            agent.epsilon = max([agent.epsilon - epsilon_decay, min_epsilon])
            mse = np.mean(episode_losses)
            history_rewards.append(total_reward)
            history_losses.append(mse)
            if not episode % print_every:
                print(
                    f'Episode: {episode}, '
                    f'rew: {np.round(total_reward, 1)}, '
                    f'mse: {np.round(mse, 2)}, '
                    f'eps: {np.round(agent.epsilon, 2)} '
                )
            break    

In [ ]:
plot_training_results(
    rewards=history_rewards,
    losses=history_losses,
    window=10,
    scale="linear"
)

In [ ]:
env = gym.make('CliffWalking-v0')
s = env.reset()
for step in itertools.count():
    print(f'STEP: {step}')
    env.render()
    action = agent.act_no_explore(s)
    s, _, done, _ = env.step(action)
    if done:
        break
env.close()

## SARSA

In [ ]:
class SARSA(QLearning):
    """
    We simply add an extra input (a1):
    From [s, a, r, s1] to --> [s, a, r, s1, a1] (sarsa)
    """
    def learn(self, s, a, r, s1, a1, done=False): 
        qsa = self.Qvalues[s][a]
        if done:
            target = r
        else:
            target = r + self.gamma * self.Qvalues[s1][a1]
           
        self.Qvalues[s][a] = (1 - self.alpha) * qsa + self.alpha * target

        return (target - qsa) ** 2


#### SARSA PSEUDOCODE

In [ ]:
env = gym.make('CliffWalking-v0')
env.reset()

nepisodes = 300
print_every = 50
gamma = 0.99
alpha = 0.1
epsilon = 1
min_epsilon = 0.05
epsilon_decay = (1 - min_epsilon) / (nepisodes * 0.5)

agent = SARSA(env, alpha=alpha, gamma=gamma, epsilon=1)

history_rewards = []
history_losses = []
for episode in range(nepisodes):
    episode_losses = []
    step = 0
    total_reward = 0
    s = env.reset()
    a = agent.act(s) # <-- Act
    while True:
        s1, r, done, _ = env.step(a)
        a1 = agent.act(s1) # <-- Next action
        total_reward += gamma**step * r
        loss = agent.learn(s, a, r, s1, a1, done)
        episode_losses.append(loss)
        a = a1 # <-- Next action becomes current action
        s = s1
        step += 1
        if done:
            agent.epsilon = max([agent.epsilon - epsilon_decay, min_epsilon])
            mse = np.mean(episode_losses)
            history_rewards.append(total_reward)
            history_losses.append(mse)
            if not episode % print_every:
                print(
                    f'Episode: {episode}, '
                    f'rew: {np.round(total_reward, 1)}, '
                    f'mse: {np.round(mse, 2)}, '
                    f'eps: {np.round(agent.epsilon, 2)} '
                )
            break    

In [ ]:
plot_training_results(
    rewards=history_rewards,
    losses=history_losses,
    window=10,
    scale="linear"
)

In [ ]:
env = gym.make('CliffWalking-v0')
s = env.reset()
for step in itertools.count():
    print(f'STEP: {step}')
    env.render()
    action = agent.act_no_explore(s)
    s, _, done, _ = env.step(action)
    if done:
        break
env.close()

## EXPECTED SARSA

In [ ]:
class ExpectedSARSA(QLearning):
    
    def learn(self, s, a, r, s1, done=False): 
        qsa = self.Qvalues[s][a]
        if done:
            target = r
        else:
            max_value = np.max(self.getQs(s1))
            rnd_value = np.mean(self.getQs(s1))
            exp_value = (1 - self.epsilon) * max_value + self.epsilon * rnd_value
            target = r + self.gamma * exp_value
        self.Qvalues[s][a] = (1 - self.alpha) * qsa + self.alpha * target

        return (target - qsa) ** 2

#### EXPECTED SARSA PSEUDOCODE -> Q-LEARNING PSEUDOCODE

In [ ]:
env = gym.make('CliffWalking-v0')
env.reset()

nepisodes = 300
print_every = 50
gamma = 0.99
alpha = 0.1
epsilon = 1
min_epsilon = 0.05
epsilon_decay = (1 - min_epsilon) / (nepisodes * 0.5)

agent = ExpectedSARSA(env, alpha=alpha, gamma=gamma, epsilon=epsilon)

history_rewards = []
history_losses = []
for episode in range(nepisodes):
    episode_losses = []
    step = 0
    total_reward = 0
    s = env.reset()
    while True:
        a = agent.act(s)
        s1, r, done, _ = env.step(a)
        total_reward += gamma**step * r
        loss = agent.learn(s, a, r, s1, done)
        episode_losses.append(loss)
        s = s1
        step += 1
        if done:
            agent.epsilon = max([agent.epsilon - epsilon_decay, min_epsilon])
            mse = np.mean(episode_losses)
            history_rewards.append(total_reward)
            history_losses.append(mse)
            if not episode % print_every:
                print(
                    f'Episode: {episode}, '
                    f'rew: {np.round(total_reward, 1)}, '
                    f'mse: {np.round(mse, 2)}, '
                    f'eps: {np.round(agent.epsilon, 2)} '
                )
            break  

In [ ]:
env = gym.make('CliffWalking-v0')
s = env.reset()
for step in itertools.count():
    print(f'STEP: {step}')
    env.render()
    action = agent.act_no_explore(s)
    s, _, done, _ = env.step(action)
    if done:
        break
env.close()